## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-13-11-49-07 +0000,nyt,Europe Haggles Over Who Will Represent It in T...,https://www.nytimes.com/2026/11/13/world/europ...
1,2026-11-13-11-38-07 +0000,nyt,"Outlook, Teams and More Microsoft Programs Dis...",https://www.nytimes.com/2026/11/12/business/mi...
2,2026-11-13-11-37-08 +0000,bbc,Nigerian university in mourning after four stu...,https://www.bbc.co.uk/news/articles/cmvg2dvw03...
3,2026-11-13-11-34-00 +0000,wsj,"Bitcoin Holds Below $70,000 as Risk-Off Mood P...",https://www.wsj.com/finance/currencies/bitcoin...
4,2026-11-13-11-27-00 +0000,wsj,Kraft Heinz Plans to Bring New Life to Old Brands,https://www.wsj.com/finance/investing/kraft-he...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2619/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
63,huang,46
56,nvidia,41
62,jensen,41
200,trump,29
33,new,25


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
375,2026-11-12-14-37-00 +0000,wsj,"From the outset, some Trump administration off...",https://www.wsj.com/politics/national-security...,213
361,2026-11-12-15-50-09 +0000,nypost,Trump floats ex-Apple CEO Tim Cook to succeed ...,https://nypost.com/2026/11/12/us-news/trump-fl...,193
113,2026-11-13-03-57-01 +0000,bbc,Ex-Nvidia CEO Jensen Huang summoned to testify...,https://www.bbc.co.uk/news/articles/cmq5vd1wdg...,192
58,2026-11-13-10-00-00 +0000,wapo,See how disgraced Nvidia CEO Jensen Huang beca...,https://www.washingtonpost.com/technology/inte...,182
291,2026-11-12-19-09-50 +0000,nypost,California Dem’s wife caught in curious Nvidia...,https://nypost.com/2026/11/12/us-news/rep-ro-k...,175


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
375,213,2026-11-12-14-37-00 +0000,wsj,"From the outset, some Trump administration off...",https://www.wsj.com/politics/national-security...
366,105,2026-11-12-15-15-00 +0000,wsj,President Trump said the U.S. doesn’t need a d...,https://www.wsj.com/politics/policy/trump-now-...
179,98,2026-11-12-23-05-02 +0000,missionlocal,"Richmond Buzz: New brunch, new pizza, and a ne...",https://missionlocal.org/2026/11/richmond-buzz...
118,65,2026-11-13-03-27-00 +0000,wsj,"Sen. Chris Van Hollen (D., Md.) said he would ...",https://www.wsj.com/politics/policy/chris-van-...
310,62,2026-11-12-18-17-00 +0000,wsj,Pentagon chief Pete Hegseth pinned a medal on ...,https://www.wsj.com/politics/hegseth-and-aides...
353,53,2026-11-12-16-29-00 +0000,wsj,"To win control of the House and Senate, Democr...",https://www.wsj.com/politics/elections/the-vot...
289,51,2026-11-12-19-12-18 +0000,nypost,Ex-Nvidia boss agrees to testify before Congre...,https://nypost.com/2026/11/12/us-news/ousted-n...
128,48,2026-11-13-02-45-20 +0000,bbc,'How does a building just fall?' Neighbours gr...,https://www.bbc.co.uk/news/articles/cm4gn5qddq...
341,46,2026-11-12-17-08-00 +0000,wsj,Home sales declined in October to their lowest...,https://www.wsj.com/economy/housing/home-sales...
309,45,2026-11-12-18-18-11 +0000,nyt,Kennedy’s Future: Will Democrats Try to Impeac...,https://www.nytimes.com/2026/11/12/us/politics...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
